#### Rename the columns and change DataTypes

In [ ]:
from pyspark.sql.functions import *

df_silver = df.withColumnRenamed("asin", "product_asin") \
              .withColumnRenamed("sentence", "sentence_text") \
              .withColumnRenamed("helpful", "helpfulness_score") \
              .withColumnRenamed("main_image_url", "product_image_url") \
              .withColumnRenamed("product_title", "product_name") \
              .withColumn( "helpfulness_score", col( "helpfulness_score").cast("double"))

#### Count the Null Values

In [ ]:
df_silver.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
]).show()

#### Trim whitespaces

In [ ]:
df_silver = df_silver.withColumn("sentence_text", trim(col("sentence_text"))) \
                     .withColumn("product_name", trim(col("product_name")))

df_silver.display()

#### Create Review Length

In [ ]:
from pyspark.sql.functions import length

df_silver = df_silver.withColumn(
    "review_length",
    length(col("sentence_text"))
)

df_silver.display()

#### Create Word Count

In [ ]:
df_silver = df_silver.withColumn(
    "word_count",
    size(split(col("sentence_text"), " "))
)
df_silver.display()

#### Create Helpfulness Category

In [ ]:
df_silver = df_silver.withColumn(
    "helpfulness_band",
    when(col("helpfulness_score") < 0.33, "Low")
    .when(col("helpfulness_score") <= 0.66, "Medium")
    .otherwise("High")
)
df_silver.display()

#### Remove Reviews with Very Few Words

In [ ]:
df_silver = df_silver.filter(
    col("word_count") >= 6
)

#### Add ingestion source and ingestion timestamp

In [ ]:
df_silver = df_silver.withColumn("ingestion_source", lit("Batch"))

In [ ]:
df_silver = df_silver.withColumn("ingestion_timestamp", current_timestamp())

#### Add sentence quality flags

In [ ]:
df_silver = df_silver.withColumn(
    "is_very_short",
    when(col("word_count") < 8, 1).otherwise(0)
).withColumn(
    "is_very_long",
    when(col("word_count") > 40, 1).otherwise(0)
).withColumn(
    "has_low_detail",
    when(
        col("sentence_text").rlike("good|nice|ok|fine") &
        (col("word_count") < 10),
        1
    ).otherwise(0)
)

#### Create Feedback Hash

In [ ]:
df_silver = df_silver.withColumn(
    "feedback_hash",
    sha2(col("sentence_text"), 256)
)

####  Window Specification

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("feedback_hash") \
                    .orderBy(col("ingestion_timestamp"))

#### Remove Duplicate Reviews

In [ ]:
df_silver = df_silver.withColumn(
    "row_num",
    row_number().over(window_spec)
)

df_silver = df_silver.filter(col("row_num") == 1) \
                     .drop("row_num", "feedback_hash")

#### Final Schema

In [ ]:
df_silver.printSchema()